# Importing Relevant Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()

from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression,  RidgeCV, LassoCV, ElasticNetCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.feature_selection import RFECV

import statsmodels.api as sm
import scipy.stats as stats
from statsmodels.stats.stattools import durbin_watson

# Importing the dataset and understanding it 

In [ ]:
df_init = pd.read_csv("Advertising.csv")
df_init.head()

## Checking for missing values and duplicates

In [ ]:
df_init.isna().sum()

In [ ]:
df_init.duplicated().sum()

Key Insights
- There are no missing values and neither are there duplicates.

In [ ]:
df_init.info()

In [ ]:
df_init.columns

In [ ]:
df1 = df_init[["TV", "Radio", "Newspaper", "Sales"]].copy()

In [ ]:
df1.shape

## Getting Descriptive Statistics 

In [ ]:
df1.describe(include="all")

Key Insights
- The counts are consistent with no missing values.
- For the TV:
    - 147.04 is spent via TV advertisement if you pick an advert at random.
- Radio:
    - 23.26 is spent via Radio advertisement if you pick an advert at random.
- Newspaper:
    - 30.55 is spent via Newspaper advertisement if you pick an advert at random.

- People spend more on Television advertisements on average as compared to Radio and Newspapers.
- The distributions of each advertisement are approximately normal given the closeness of the means and the respective medians. Radio and Newspaper adverts show signs of right skewness and probable presence of extreme values.

## Checking for outliers 

In [ ]:
plt.figure(figsize=(12,6))
sns.boxplot(df1)
plt.title("Box plot of the features")
plt.xlabel("")
plt.ylabel("")

In [ ]:
Q1 = df1["Newspaper"].quantile(0.25)
Q3 = df1["Newspaper"].quantile(0.75)
IQR = Q3-Q1

upper_bound = Q3 + 1.5*IQR

upper_bound

In [ ]:
newpaper_outliers = df1[df1["Newspaper"]>upper_bound]

newpaper_outliers

### Using the winsorizing technique to cap the outliers 

In [ ]:
df = df1.copy()

df["Newspaper"] = np.where(
    df1["Newspaper"]>upper_bound,
    upper_bound,
    df["Newspaper"]
)

## Plotting pairplots 

In [ ]:
plt.figure(figsize=(12,8))
sns.pairplot(df)
plt.show()

Key insights
- There is a positive linear relationship between sales and TV, Radio, and Newspaper advertisement.
- The more the TV adverts, the more the sales. This is the same for Radio adverts.
- The Radio, Newspapers and TV sales have an inconsistent variance forming funnels.
- Newspaper adverts on the other hand have very low linear relationship with the sales.

## Scatter plots

### Scatter Plot for sales vs TV, Radio, and Newspaper adverts 

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12,4))

sns.scatterplot(data=df, x="Sales", y="TV", ax=axes[0])
axes[0].set_title("sales vs TV advertisement")

sns.scatterplot(data=df, x="Sales", y="Radio", ax=axes[1])
axes[1].set_title("sales vs Radio advertisement")

sns.scatterplot(data=df, x="Sales", y="Newspaper", ax=axes[2])
axes[2].set_title("sales vs Newspaper advertisement")

plt.tight_layout()
plt.show()

Key insights
- The return on advertising spend is clearly strongest for TV and Radio, where higher spend visibly tracks with higher Sales.
- Newspaper shows a much weaker, noisier relationship with Sales.

## Heatmap

In [ ]:
cor_matrix = df.corr()

In [ ]:
plt.figure(figsize=(14,4))
sns.heatmap(
    cor_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    linewidth=1,
    cbar=True
)
plt.title("Correlation Matrix heatmap")
plt.tight_layout()
plt.show()

- This confirms the low linear relationship with the newspaper adverts

# Fitting the linear model

## 80/20 split

In [ ]:
X = df.drop(columns=["Sales"])
y = df["Sales"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=256
)

lr = LinearRegression()
lr = lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)

In [ ]:
lr_mae = mean_absolute_error(y_test, lr_pred)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred))
lr_r2 = r2_score(y_test, lr_pred)

In [ ]:
print(f"R_squared : {lr_r2:.4f}")
print(f"RMSE : {lr_rmse:.4f}")
print(f"MAE : {lr_mae:.4f}")

## Checking model assumptions

### Linearity

In [ ]:
y_train_pred = lr.predict(X_train)
residuals = y_train - y_train_pred

In [ ]:
plt.figure(figsize=(10,6))
sns.scatterplot(x=y_train_pred, y=residuals)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs fitted values")
plt.show() 

### Normality of residuals 

In [ ]:
plt.figure(figsize=(10,5))
stats.probplot(residuals, dist="norm", plot=plt)
plt.title("Q-Q Plot of residuals")
plt.show()

In [ ]:
stat, p_value = stats.shapiro(residuals)
print(f"Shapiro-Wilk Statistic: {stat:.4f}, P_value: {p_value:.4f}")

- The residuals are not normally distributed.
- The downward bending shows heteroscedasticity and evidence of a non linear relationship.
- This suggests transformation may be needed or use of polynomial terms

### Multicollinearity (VIF) 

In [ ]:
vif_data = pd.DataFrame()
vif_data["Feature"] = X.columns
vif_data["VIF"] = [
    variance_inflation_factor(X.values, i) for i in range(X.shape[1])
]

print(vif_data)

There is no evidence of Multicollinearity

### Independence of residuals 

In [ ]:
dw_stat = durbin_watson(residuals)
print(f"Durbin-Watson Statistic: {dw_stat:.4f}")

The residuals are independent

## Remodelling with interaction terms after centering including polynomial terms of at most degree 2. (Polynomial Regression)

In [ ]:
poly_inter = PolynomialFeatures(degree=2, include_bias=False)

In [ ]:
scaler =StandardScaler(with_mean=True, with_std=True)

X_train_centered = scaler.fit_transform(X_train)
X_test_centered = scaler.transform(X_test)

X_train_inter = poly_inter.fit_transform(X_train_centered)
X_test_inter = poly_inter.transform(X_test_centered)

lr_inter =LinearRegression()
lr_inter.fit(X_train_inter, y_train)

lr_inter_pred = lr_inter.predict(X_test_inter)

In [ ]:
mae_inter = mean_absolute_error(y_test, lr_inter_pred)
rmse = np.sqrt(mean_squared_error(y_test, lr_inter_pred))
rs_inter = r2_score(y_test, lr_inter_pred)

In [ ]:
print(f"R_squared : {rs_inter:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAE : {mae_inter:.4f}")

In [ ]:
feature_names = poly_inter.get_feature_names_out(X.columns)
print(feature_names)

In [ ]:
residuals_inter = y_test - lr_inter_pred

## Rechecking the assumptions 

### Linearity

In [ ]:
plt.figure(figsize=(10,6))
sns.scatterplot(x=lr_inter_pred, y=residuals_inter)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs fitted values")
plt.show() 

### Normality

In [ ]:
plt.figure(figsize=(10,5))
stats.probplot(residuals_inter, dist="norm", plot=plt)
plt.title("Q-Q Plot of residuals")
plt.show()

In [ ]:
stat, p_value = stats.shapiro(residuals_inter)
print(f"Shapiro-Wilk Statistic: {stat:.4f}, P_value: {p_value:.4f}")

- The residuals are normally distributed.

### Multicollinearity

In [ ]:
vif_inter = pd.DataFrame()
vif_inter["Feature"] = feature_names
vif_inter["VIF"] = [
    variance_inflation_factor(X_train_inter, i)
    for i in range(X_train_inter.shape[1])
]

print(vif_inter)

- There is no evidence of multicollinearity

## Generating a similar model output to statsmodel

In [ ]:
# Convert the transformed numpy array back to a DataFrame
X_train_inter_df = pd.DataFrame(X_train_inter, columns=feature_names, index=X_train.index)

# Add a constant
X_train_sm = sm.add_constant(X_train_inter_df)

ols_model = sm.OLS(y_train, X_train_sm).fit()

print(ols_model.summary())

## Regularization for feature selection

In [ ]:
scaler1 = StandardScaler()

X_train_scaled = scaler1.fit_transform(X_train_inter)
X_test_scaled = scaler1.transform(X_test_inter)

### L2 Regularization - Ridge

In [ ]:
ridge = RidgeCV(cv=5)
ridge.fit(X_train_scaled, y_train)

for feature, coef in zip(feature_names, ridge.coef_):
    if coef != 0:
        print(f"{feature}: {coef:.4f}")

### L1 Regularization - Lasso

In [ ]:
lasso = LassoCV(cv=5, random_state=256)
lasso.fit(X_train_scaled, y_train)

for feature, coef in zip(feature_names, lasso.coef_):
    if coef != 0:
        print(f"{feature}: {coef:.4f}")

### L1 + L2 - Elastic Net 

In [ ]:
elastic = ElasticNetCV(l1_ratio=[0.1, 0.5, 0.7, .9, 0.99], cv=5, random_state=256)
elastic.fit(X_train_scaled, y_train)

for feature, coef in zip(feature_names, elastic.coef_):
    if coef != 0:
        print(f"{feature}: {coef:.4f}")

### RFECV 

In [ ]:
rfecv = RFECV(LinearRegression(), step=1, cv=5, scoring="r2")
rfecv.fit(X_train_scaled, y_train)

for feature in feature_names[rfecv.support_]:
    print(feature)

In [ ]:
results = pd.DataFrame({
    "Model": ["Ridge", "Lasso", "Elastic Net", "RFECV"],
    "Test R2": [
        ridge.score(X_test_scaled, y_test),
        lasso.score(X_test_scaled, y_test),
        elastic.score(X_test_scaled, y_test),
        rfecv.score(X_test_scaled, y_test)
    ],
    "Features Kept": [
        sum(ridge.coef_ != 0),
        sum(lasso.coef_ != 0),
        sum(elastic.coef_ != 0),
        rfecv.n_features_
    ]
})


results = results.sort_values(by="Test R2", ascending=False)

print("Model Comparison")
print(results.to_string(index=False))

Key Insights
- The initial standard Linear Regression violated linearity and normality assumptions because it failed to capture the complex, non-linear behavior of advertising spend.
- Engineering polynomial and interaction terms corrected the residual assumptions but introduced severe structural multicollinearity and statistically insignificant noise, specifically from Newspaper interactions.
- Recursive Feature Elimination (RFECV) successfully identified and removed the statistical noise, yielding an optimized model with a high test $R^2$ of approximately 0.98. This model was also chosen because it was coherent with the statistical significance of p_values.
- Channel synergy is a critical driver of sales; the interaction between TV and Radio indicates that simultaneous campaigns produce a multiplicative effect, generating more sales than treating the channels independently.
- TV advertising exhibits diminishing marginal returns, meaning it is highly effective but eventually hits a saturation point where additional spend yields slightly smaller returns.
- Newspaper advertisement consistently proved to be statistically insignificant and ineffective in driving sales, suggesting its budget could be reallocated to TV and Radio for higher ROI.

## Final Linear Model

In [ ]:
optimal_features = feature_names[rfecv.support_]

In [ ]:
X_train_final = pd.DataFrame(X_train_scaled, columns=feature_names)[optimal_features]
X_test_final = pd.DataFrame(X_test_scaled, columns=feature_names)[optimal_features]

In [ ]:
X_train_final_sm = sm.add_constant(X_train_final.values)
final_model = sm.OLS(y_train, X_train_final_sm).fit()

final_fitted_values = final_model.fittedvalues
final_residuals = final_model.resid

print(final_model.summary(xname=["const"] + list(optimal_features)))

In [ ]:
# Predict using the optimized RFECV model on the scaled test data
rfecv_pred = rfecv.predict(X_test_scaled)

# Calculate the evaluation metrics
rfecv_mae = mean_absolute_error(y_test, rfecv_pred)
rfecv_rmse = np.sqrt(mean_squared_error(y_test, rfecv_pred))
rfecv_r2 = r2_score(y_test, rfecv_pred)

# Print the results matching your previous format
print(f"R_squared : {rfecv_r2:.4f}")
print(f"RMSE : {rfecv_rmse:.4f}")
print(f"MAE : {rfecv_mae:.4f}")

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x=final_fitted_values, y=final_residuals)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Fitted Values (Final Polynomial Model)")
plt.ylabel("Residuals")
plt.title("Residuals vs Fitted Values: Final Polynomial Model")
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
stats.probplot(final_residuals, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals: Final Polynomial Model")
plt.show()

In [ ]:
stat, p_value = stats.shapiro(final_residuals)
print(f"Shapiro-Wilk Statistic: {stat:.4f}, P-value: {p_value:.4f}")

## Investigating model outliers

In [ ]:
# Check training residuals since that's where the full-model plot was generated
train_residuals = final_model.resid

# Find training outliers
train_investigation = X_train.copy()
train_investigation['Actual_Sales'] = y_train
train_investigation['Predicted_Sales'] = final_model.fittedvalues
train_investigation['Residual'] = train_residuals


outliers = train_investigation[train_investigation['Residual'] < -1.5]
print(outliers)

- After investigating the extreme negative residuals (such as Index 155 and Index 130), it is evident that these specific points represent isolated market anomalies—instances where sales plummeted despite active TV and Radio advertising spend.
- Because Ordinary Least Squares (OLS) regression globally minimizes squared errors across every data point, these severe downward shocks forced the regression line to shift, pulling on the error distribution tails and causing the Shapiro-Wilk test to flag non-normality.
- While the optimized polynomial and RFECV models successfully captured the core market dynamics and achieved a high out-of-sample $R^2$ of ~0.98, parametric linear models remain sensitive to unmeasured external disruptions. This justifies the parallel use of tree-based algorithms like the Random Forest, which naturally isolate such irregularities into separate leaf nodes without letting them distort overall model performance.

### Model Performance Summary

| Model | R-squared | RMSE | MAE | Features Used |
|---|---|---|---|---|
| Base Linear Regression | 0.8461 | 1.7672 | 1.2978 | 3 (Raw main effects) |
| Full Polynomial Regression | 0.9867 | 0.5198 | 0.4349 | 9 (Included all noise) |
| Optimized RFECV Model | 0.9859 | 0.5355 | 0.4518 | Optimal subset (Dropped noise) |

Key Insights

- Complex Dynamics Drive Sales: By introducing polynomial and interaction terms, I observed a **massive leap in R²** (from 0.84 to nearly 0.99) and a **~70% drop in prediction errors** between my Base and Polynomial models. This proves that advertising channels do not operate in a straight, independent line. Capturing the **synergy between TV and Radio**, along with the **saturation point of TV**, is mandatory for accurate forecasting.
- Accuracy Does Not Require Complexity: My Full Polynomial model achieved its 0.9867 score by hoarding every possible interaction, including statistically insignificant dead weight like the Newspaper interactions. By applying **RFECV**, I maintained virtually the exact same predictive power (**R² of 0.9859**) while successfully stripping away those redundant features.
- The Parsimony Advantage: In choosing the RFECV model, I successfully navigated the **bias-variance tradeoff**. I secured the high accuracy of a complex model while retaining a simple, stable, and highly interpretable equation that can be confidently used to allocate the business advertising budget.
- Both channels have a strong, positive linear impact on sales. TV is the dominant individual driver of sales, contributing more per standard deviation increase than Radio.
- The negative coefficient on the quadratic term mathematically confirms the saturation point of TV campaigns. While TV is the strongest driver, the negative pull of this squared term means that as TV expenditure gets exceptionally high, the curve flattens and each additional dollar spent on TV yields progressively fewer sales.
- The positive coefficient on the interaction term proves that TV and Radio do not operate in silos. Running simultaneous campaigns on both channels creates a multiplicative effect, generating 1.34 more units of sales per standardized unit than if you had allocated the budget to just one channel independently.

# Fitting the Random Forest Regressor

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=256)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

In [ ]:
rf_mae = mean_absolute_error(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_r2 = r2_score(y_test, rf_pred)

In [ ]:
print(f"R_squared : {rf_r2:.4f}")
print(f"RMSE : {rf_rmse:.4f}")
print(f"MAE : {rf_mae:.4f}")

- This is performing exceptionally well immediately. Cross Validation is next to curb any anomalies.

## Hyperparameter tuning

In [ ]:
rf_base = RandomForestRegressor(random_state=256)

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10, 15],
    'min_samples_split': [2, 5, 10]
}

grid_search = GridSearchCV(
    estimator=rf_base, 
    param_grid=param_grid, 
    cv=5, 
    scoring='r2', 
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

In [ ]:
best_rf = grid_search.best_estimator_
best_rf_pred = best_rf.predict(X_test)

In [ ]:
best_rf_mae = mean_absolute_error(y_test, best_rf_pred)
best_rf_rmse = np.sqrt(mean_squared_error(y_test, best_rf_pred))
best_rf_r2 = r2_score(y_test, best_rf_pred)

print("Cross-Validation and Grid Search Results")
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Average CV R_squared: {grid_search.best_score_:.4f}")
print(f"Test R_squared : {best_rf_r2:.4f}")
print(f"Test RMSE : {best_rf_rmse:.4f}")
print(f"Test MAE : {best_rf_mae:.4f}")

- There is a high generalization since the Random Forest is very close to the test $R^2$, hence it is not entirely relying on luck.
- The optimal tree depth chosen is 10.
- The maximum number of trees were chosen for the best outcome.

## Feature Importance 

In [ ]:
importances = best_rf.feature_importances_
feature_names = X.columns

In [ ]:
rf_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=rf_importance_df, x='Importance', y='Feature')
plt.title("Random Forest Feature Importances")
plt.show()

- TV is the Dominant Driver (~62% Importance). TV carries the vast majority of the predictive weight for the sales. It is the absolute core of the marketing strategy.
- Radio is a Strong Secondary Channel (~37% Importance). While not as powerful as TV alone, Radio provides massive predictive value.
- Newspaper is Dead Weight (< 1% Importance). The Random Forest essentially ignored Newspaper advertising entirely when building its decision trees to predict sales.

## Analyzing residuals of the Random Forest 

In [ ]:
rf_residuals = y_test - best_rf_pred

plt.figure(figsize=(10, 6))
sns.scatterplot(x=best_rf_pred, y=rf_residuals)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Fitted Values (Random Forest)")
plt.ylabel("Residuals")
plt.title("Random Forest Residuals vs Fitted Values")
plt.show()

- The predictions are unbiased, the errors are random noise

# Conclusions

Throughout this analysis of the advertising dataset, we evaluated multiple modeling approaches—from baseline Ordinary Least Squares (OLS) regression and regularized linear models (Ridge, Lasso, Elastic Net) to polynomial interaction terms and tree-based ensemble methods (Random Forest).

## Core Advertising Insights

**TV is the Dominant Channel:** Television advertising carries the vast majority of predictive weight (~62% importance in tree models) and exhibits a powerful non-linear saturation curve that dictates baseline revenue.

**Radio Acts as a Crucial Synergy Driver:** While secondary to TV in isolation, Radio provides vital support and exhibits a strong interaction effect that accelerates overall returns.

**Newspaper is Ineffective:** Across every feature selection method (RFECV, Lasso, and Random Forest), Newspaper advertising contributed negligible predictive power (<1%), confirming it should be eliminated from the budget.

## Model Performance & Diagnostics

- The initial linear model suffered from high bias and a distinct U-shaped residual pattern, missing underlying non-linearities and failing normality tests (Shapiro-Wilk p-value of 0.0000).

- Incorporating polynomial terms ($\text{TV}^2$ and $\text{TV} \times \text{Radio}$) significantly improved out-of-sample accuracy, pushing $R^2$ to roughly 0.98. However, extreme residual outliers (such as instances where market shocks suppressed sales despite active advertising) continued to strain the normality assumptions of the parametric linear framework.

- The Random Forest Regressor naturally resolved these non-linear dynamics and isolated anomalous data points into separate leaf nodes without requiring manual feature engineering or breaking distributional assumptions, confirming it as a robust alternative.

## Final Recommendation

Stakeholders should immediately reallocate 100% of the underperforming Newspaper budget into TV and Radio campaigns. For strategic planning and budget simulation, the optimized polynomial model provides an interpretable mathematical formula, while the Random Forest offers maximum predictive reliability across volatile market conditions.